# TerraTree (Kali track) — Step 3: Model Training (Family Classifier + Biodiversity Regressor)

With n=56, a standard 70/30 split isn't statistically sound (some classes
have as few as 3 samples — a random split could easily leave a class with
0 samples in the test set). Instead we use **Leave-One-Out Cross-Validation
(LOOCV)**: train on 55 plots, test on the 1 held out, repeat 56 times.
This is the textbook-correct approach for small-n problems and is itself
worth stating explicitly in your report as a deliberate methodological choice.

In [ ]:
from google.colab import files
import pandas as pd
import numpy as np

uploaded = files.upload()  # upload kali_training_table.csv
df = pd.read_csv('kali_training_table.csv')
print(df.shape)
df.head()

In [ ]:
feature_cols = [c for c in df.columns if c not in
                ['plot_id', 'decimalLatitude', 'decimalLongitude',
                 'dominant_family', 'species_richness', 'shannon_index']]
print("Features:", feature_cols)

X = df[feature_cols]

## Model 1: Dominant plant family classifier (LOOCV)

In [ ]:
from sklearn.model_selection import LeaveOneOut
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_family = df['dominant_family']

loo = LeaveOneOut()
y_true_all, y_pred_all = [], []

for train_idx, test_idx in loo.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y_family.iloc[train_idx], y_family.iloc[test_idx]

    clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)

    y_true_all.extend(y_test.tolist())
    y_pred_all.extend(pred.tolist())

oa = accuracy_score(y_true_all, y_pred_all)
print(f"LOOCV Overall Accuracy: {oa:.4f}  ({len(y_true_all)} plots)\n")
print(classification_report(y_true_all, y_pred_all, zero_division=0))

In [ ]:
import matplotlib.pyplot as plt

labels = sorted(y_family.unique())
cm = confusion_matrix(y_true_all, y_pred_all, labels=labels)

plt.figure(figsize=(7, 6))
plt.imshow(cm, cmap='Greens')
plt.xticks(range(len(labels)), labels, rotation=90)
plt.yticks(range(len(labels)), labels)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Dominant Family — LOOCV Confusion Matrix')
for i in range(len(labels)):
    for j in range(len(labels)):
        plt.text(j, i, cm[i, j], ha='center', va='center',
                  color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.colorbar()
plt.tight_layout()
plt.show()

## Model 2: Biodiversity regression (species richness + Shannon index)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

for target_name in ['species_richness', 'shannon_index']:
    y = df[target_name]
    preds = []

    for train_idx, test_idx in loo.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train = y.iloc[train_idx]

        reg = RandomForestRegressor(n_estimators=200, random_state=42)
        reg.fit(X_train, y_train)
        preds.append(reg.predict(X_test)[0])

    r2 = r2_score(y, preds)
    mae = mean_absolute_error(y, preds)
    print(f"{target_name}: LOOCV R² = {r2:.3f}, MAE = {mae:.3f}")

    plt.figure(figsize=(5, 5))
    plt.scatter(y, preds, alpha=0.7)
    plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', label='Perfect prediction')
    plt.xlabel(f'Actual {target_name}')
    plt.ylabel(f'Predicted {target_name}')
    plt.title(f'{target_name}: Actual vs Predicted (LOOCV)')
    plt.legend()
    plt.tight_layout()
    plt.show()

## Train final models on ALL 56 plots (for the dashboard)

LOOCV above is for honest evaluation. For the actual deployed model (what
the dashboard uses to make predictions on new points), train once more on
the full dataset — no need to hold anything out once evaluation is done.

In [ ]:
final_family_clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
final_family_clf.fit(X, y_family)

final_richness_reg = RandomForestRegressor(n_estimators=200, random_state=42)
final_richness_reg.fit(X, df['species_richness'])

final_shannon_reg = RandomForestRegressor(n_estimators=200, random_state=42)
final_shannon_reg.fit(X, df['shannon_index'])

import joblib
bundle = {
    'family_classifier': final_family_clf,
    'richness_regressor': final_richness_reg,
    'shannon_regressor': final_shannon_reg,
    'features': feature_cols,
    'family_classes': sorted(y_family.unique()),
    'loocv_family_accuracy': oa,
    'n_plots': len(df),
}
joblib.dump(bundle, 'kali_models.joblib')
files.download('kali_models.joblib')
print("Models saved and downloading.")

## Next steps
- [ ] Report the LOOCV accuracy/R² honestly, with the n=56 caveat stated plainly — this is real ground truth, but a small sample, and LOOCV is the statistically appropriate choice for that, not a weakness to hide
- [ ] Check the confusion matrix — expect the well-populated classes (Lauraceae, Rubiaceae, Fabaceae) to perform better than the sparse "Other family" catch-all
- [ ] Build the Kali dashboard (or extend the existing Streamlit app with a second tab/page for this track)